Solution to [Day 6 Cephalod Math](https://adventofcode.com/2025/day/6).

Part 1. Sum or product of columns. I used the polars module in Python to create and handle a dataframe of numbers and one row of operators (* or -). The function *get_total_of_sums_and_products* loops through every column and adds the total of all operations.

Part 2. Numbers are written differently, the spaces in the text input become important as the digits of a number are ranged vertically. The polars dataframe used therefore has string data, keeping spaces where they are, this is done through function *helper_get_df_of_numstrings*. 

Function *helper_numstr_list_to_cephalod_nums* converts columns of numstrings to integers using the cephalod way. Operations and totals are done through *get_total_of_sums_and_products_pt2*. 

-Annette

In [55]:
import polars as pl  #Part 1 and 2
import math #Part 2

In [56]:
# Part 1 Get data
with open('inputs/day6_input.txt','r') as f: 
    lines = f.read().splitlines() #list of strings

linesv2 = [l.split(' ') for l in lines] #each line becomes a list of integers and ''
linesv3 = [[int(e) for e in l if e.isdigit()] for l in linesv2[:4]] #each line is a list of integers only

#get operators * or - as a separate list
operators = [op for op in linesv2[4] if op != '']

#convert list into a polars dataframe of the integers
df = pl.DataFrame(linesv3, orient='row')

In [57]:
#Part 1 Function

def get_total_of_sums_and_products(df:pl.DataFrame, operators) -> int:
    '''return dataframe totals after doing the designated operation * or -
    on each column
    Returns an integer as total'''
    df_total = 0
    for i in range(len(df.columns)):
        operator = operators[i]
        # print(i, operator)
        if operator == '+':
            df_total += df[:,i].sum()
        elif operator == '*':
            df_total += df[:,i].product()
    return df_total


In [58]:
# Part 1 Solution

print(f"Part 1 Answer: {get_total_of_sums_and_products(df, operators)}")

Part 1 Answer: 5877594983578


In [59]:
#Part 2 Functions

def helper_get_df_of_numstrings(lines:list[str])->pl.dataframe:
    '''split each list of intput strings (from the txt file)into a dataframe.
    For Part2, the spaces around the integers cannot be ignored as they determine 
    which vertical column of digits are combined to form chephalod numbers'''
    
    #get the column borders, for all rows these borders would have ' ' as the value
    space_indices = [i for i in range(len(lines[0])) if all([line[i]== ' ' for line in lines])]
    linesv2 = []

    for line in lines:
        templine = []
        start_index = 0
        for i in space_indices: #first n-1 columns
            end_index = i
            templine.append(line[start_index:end_index])
            start_index = i+1
        templine.append(line[start_index:]) #last column
        linesv2.append(templine)
    return pl.DataFrame(linesv2, orient='row')

def helper_numstr_list_to_cephalod_nums(numlist:list[str]) -> list[int]:
    '''converts each column of number strings into a list of cephalod numbers.
    the list vary in length (depending on the max number of digits in each column).'''
    max_str_len = len(numlist[0]) #all strings have same length, some have more spaces
    vertical_ints = []
    for i in range(max_str_len):
    #(-1,max_str_len-1,-1): #indices going from right to left
        vert_num_str = ''
        for ns in numlist:
            vert_num_str += ns[i]
        vert_num_str = vert_num_str.replace(' ','') #spaces removed here
        vertical_ints.append(int(vert_num_str))
    return vertical_ints

def get_total_of_sums_and_products_pt2(df:pl.dataframe, operators: list[str]) -> int:
    '''in Part2, number list is created differently and uses the helper function above.
    This function uses a polars dataframe of strings (of digits) instead of integers
    and the list of operators.
    Returns the sum of all the operations on cephalod numbers'''
    df_total = 0
    for i in range(len(df.columns)):
        cephalod_ints = helper_numstr_list_to_cephalod_nums(df[:,i].to_list())
        operator = operators[i]
        if operator == '+':
            df_total += sum(cephalod_ints)
        elif operator == '*':
            df_total += math.prod(cephalod_ints)
    return df_total
    


In [60]:
#Part 2 Solution

df2 = helper_get_df_of_numstrings(lines[:4])

print(f"Part 2 Answer: {get_total_of_sums_and_products_pt2(df2, operators)}")

Part 2 Answer: 11159825706149
